## Construction Dataset Generator

Generates a synthetic **construction** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `construction` | `projects` | ~2K | `work_orders` | 100K-500K | 12 project types (bridges, stadiums, tunnels, hospitals), cost overrun modeling, seasonal weather delays, safety incident tracking, change order patterns |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `construction` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.

In [0]:
dbutils.widgets.text("catalog", "industry_sample_data")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.construction') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.construction');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(77)
random.seed(77)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "construction"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime(2026, 3, 26)

def clamp(val, lo, hi):
  return float(max(lo, min(hi, val)))

# --- Projects table (~2,000 rows) ---

project_types = ["Bridge", "Stadium", "Tunnel", "Hospital", "Highway", "High-Rise",
                 "Airport Terminal", "Dam", "Rail Station", "Convention Center",
                 "School", "Water Treatment Plant"]
project_type_weights = [10, 5, 6, 12, 15, 12, 4, 3, 8, 5, 14, 6]

# Contract value parameters (mu, sigma for log-normal, lo, hi in millions)
contract_params = {
  "Bridge":                (4.5, 0.8,   20,  2000),
  "Stadium":               (5.5, 0.7,  150,  5000),
  "Tunnel":                (5.0, 0.9,   80,  3000),
  "Hospital":              (4.8, 0.7,   30,  2000),
  "Highway":               (4.0, 0.9,   10,  1000),
  "High-Rise":             (5.0, 0.7,   80,  2000),
  "Airport Terminal":      (5.5, 0.8,  150,  5000),
  "Dam":                   (4.8, 0.9,   40,  3000),
  "Rail Station":          (4.3, 0.7,   30,  1000),
  "Convention Center":     (4.8, 0.7,   80,  2000),
  "School":                (3.0, 0.6,    8,   200),
  "Water Treatment Plant": (4.0, 0.7,   15,   500),
}

# Duration parameters (min_months, max_months) by project type
duration_params = {
  "Bridge":                (24, 84),
  "Stadium":               (36, 72),
  "Tunnel":                (36, 120),
  "Hospital":              (24, 60),
  "Highway":               (12, 60),
  "High-Rise":             (24, 72),
  "Airport Terminal":      (36, 96),
  "Dam":                   (36, 96),
  "Rail Station":          (24, 60),
  "Convention Center":     (24, 60),
  "School":                (12, 36),
  "Water Treatment Plant": (24, 60),
}

# Peak workforce by project type (mu, sigma for Gaussian)
workforce_params = {
  "Bridge":                (800, 250),
  "Stadium":               (2500, 700),
  "Tunnel":                (1200, 400),
  "Hospital":              (1500, 400),
  "Highway":               (600, 200),
  "High-Rise":             (1800, 500),
  "Airport Terminal":      (3000, 800),
  "Dam":                   (1000, 300),
  "Rail Station":          (800, 250),
  "Convention Center":     (1500, 400),
  "School":                (200, 80),
  "Water Treatment Plant": (400, 150),
}

# Site area by project type (mu, sigma for log-normal in sqm)
site_area_params = {
  "Bridge":                (10.0, 0.7),
  "Stadium":               (11.0, 0.5),
  "Tunnel":                (9.5,  0.8),
  "Hospital":              (10.2, 0.5),
  "Highway":               (11.5, 0.8),
  "High-Rise":             (8.5,  0.5),
  "Airport Terminal":      (11.5, 0.5),
  "Dam":                   (11.0, 0.7),
  "Rail Station":          (9.5,  0.5),
  "Convention Center":     (10.5, 0.5),
  "School":                (9.0,  0.4),
  "Water Treatment Plant": (9.8,  0.5),
}

regions = {
  "Northeast":         ["New York", "Boston", "Philadelphia", "Pittsburgh", "Hartford", "Newark"],
  "Southeast":         ["Atlanta", "Miami", "Charlotte", "Nashville", "Tampa", "Orlando"],
  "Midwest":           ["Chicago", "Detroit", "Minneapolis", "Columbus", "Indianapolis", "Milwaukee"],
  "Southwest":         ["Dallas", "Houston", "Phoenix", "San Antonio", "Austin", "Denver"],
  "West":              ["Los Angeles", "San Francisco", "San Diego", "Las Vegas", "Sacramento"],
  "Pacific Northwest": ["Seattle", "Portland", "Boise", "Spokane"],
}
region_weights = [22, 20, 18, 20, 15, 5]
region_names = list(regions.keys())

project_name_descriptors = {
  "Bridge":                ["Crossing", "Span", "Overpass", "Viaduct", "Connector"],
  "Stadium":               ["Arena", "Sports Complex", "Athletic Center", "Field"],
  "Tunnel":                ["Tunnel", "Underground Passage", "Bore", "Subway Extension"],
  "Hospital":              ["Medical Center", "Hospital", "Health Campus", "Medical Pavilion"],
  "Highway":               ["Expressway", "Interstate Extension", "Highway Widening", "Freeway"],
  "High-Rise":             ["Tower", "Plaza", "Center", "Skyscraper"],
  "Airport Terminal":      ["Terminal", "Concourse", "Airport Expansion", "Air Hub"],
  "Dam":                   ["Dam", "Reservoir", "Water Control", "Flood Control"],
  "Rail Station":          ["Station", "Transit Hub", "Rail Terminal", "Metro Stop"],
  "Convention Center":     ["Convention Center", "Expo Hall", "Conference Center", "Event Center"],
  "School":                ["Academy", "School", "Campus", "Learning Center"],
  "Water Treatment Plant": ["Water Plant", "Treatment Facility", "Filtration Plant", "Water Works"],
}

NUM_PROJECTS = 2000
projects = []
for i in range(1, NUM_PROJECTS + 1):
  proj_type = random.choices(project_types, weights=project_type_weights)[0]
  region = random.choices(region_names, weights=region_weights)[0]
  city = random.choice(regions[region])

  descriptor = random.choice(project_name_descriptors[proj_type])
  phase_suffix = f" Phase {random.randint(1, 4)}" if random.random() < 0.3 else ""
  project_name = f"{city} {descriptor}{phase_suffix}"

  client_name = fake.company()
  project_manager = fake.name()

  # Contract value (log-normal, in millions, then convert to dollars)
  cp = contract_params[proj_type]
  contract_value = round(clamp(random.lognormvariate(cp[0], cp[1]), cp[2], cp[3]) * 1_000_000, -3)

  # Start date: between 2018 and 2025
  start_date = date(2018, 1, 1) + timedelta(days=random.randint(0, 2800))

  dp = duration_params[proj_type]
  duration_months = random.randint(dp[0], dp[1])
  est_completion = start_date + timedelta(days=duration_months * 30)

  days_since_start = (NOW.date() - start_date).days
  days_to_est_completion = (est_completion - NOW.date()).days

  # Status depends on timeline relative to NOW
  if est_completion < NOW.date() - timedelta(days=180):
    status = random.choices(["Completed", "In Progress", "On Hold", "Cancelled"],
                            weights=[80, 3, 5, 12])[0]
  elif start_date > NOW.date():
    status = random.choices(["Planning", "On Hold", "Cancelled"],
                            weights=[80, 12, 8])[0]
  elif days_to_est_completion > 0:
    status = random.choices(["In Progress", "On Hold", "Planning", "Cancelled"],
                            weights=[70, 15, 10, 5])[0]
  else:
    status = random.choices(["Completed", "In Progress", "On Hold"],
                            weights=[55, 35, 10])[0]

  # Actual completion date (with overrun tendency)
  if status == "Completed":
    overrun_days = int(clamp(random.gauss(duration_months * 3, duration_months * 5),
                             -30, duration_months * 15))
    actual_completion = est_completion + timedelta(days=overrun_days)
    if actual_completion > NOW.date():
      actual_completion = NOW.date() - timedelta(days=random.randint(1, 60))
  else:
    actual_completion = None

  # Completion percentage (correlated with status and timeline)
  if status == "Completed":
    completion_pct = 100.0
  elif status == "Cancelled":
    completion_pct = round(clamp(random.betavariate(2.0, 5.0) * 100, 0, 60), 1)
  elif status == "Planning":
    completion_pct = round(clamp(random.uniform(0, 5), 0, 5), 1)
  elif status == "On Hold":
    completion_pct = round(clamp(random.betavariate(2.0, 3.0) * 100, 5, 85), 1)
  else:
    if days_to_est_completion > 0 and days_since_start > 0:
      expected_progress = days_since_start / (days_since_start + days_to_est_completion)
      completion_pct = round(clamp(
        expected_progress * 100 * clamp(random.gauss(1.0, 0.15), 0.6, 1.3), 5, 95), 1)
    else:
      completion_pct = round(clamp(random.betavariate(5.0, 1.5) * 100, 70, 98), 1)

  # Safety rating (0-100, beta distributed, higher is better)
  base_safety = clamp(random.betavariate(6.0, 2.0) * 100, 40, 100)
  if proj_type in ["Tunnel", "Dam", "Bridge"]:
    base_safety *= random.uniform(0.85, 0.95)
  safety_rating = round(clamp(base_safety, 40, 100), 1)

  # Environmental impact score (0-100, lower is better)
  env_base = clamp(random.betavariate(2.5, 3.5) * 100, 5, 95)
  if proj_type in ["Highway", "Dam", "Tunnel"]:
    env_base = clamp(env_base * 1.3, 20, 95)
  elif proj_type in ["School", "Hospital"]:
    env_base = clamp(env_base * 0.8, 5, 70)
  environmental_impact_score = round(env_base, 1)

  wp = workforce_params[proj_type]
  num_workers_peak = int(clamp(random.gauss(wp[0], wp[1]), 50, 8000))

  sap = site_area_params[proj_type]
  site_area = round(clamp(random.lognormvariate(sap[0], sap[1]), 500.0, 500000.0), 0)

  projects.append(Row(
    project_id=i,
    project_name=project_name,
    project_type=proj_type,
    client_name=client_name,
    project_manager=project_manager,
    region=region,
    city=city,
    status=status,
    contract_value=contract_value,
    start_date=start_date,
    estimated_completion_date=est_completion,
    actual_completion_date=actual_completion,
    completion_pct=completion_pct,
    safety_rating=safety_rating,
    environmental_impact_score=environmental_impact_score,
    num_workers_peak=num_workers_peak,
    site_area_sqm=site_area
  ))

project_lookup = {p.project_id: p for p in projects}

projects_df = spark.createDataFrame(projects)
projects_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.projects")
print(f"\u2714 Created {CATALOG_SCHEMA}.projects ({projects_df.count()} rows)")

# --- Work Orders table (randomized ~100K-500K rows) ---

work_order_types = [
  "Excavation", "Demolition", "Grading", "Piling", "Foundation Pour",
  "Structural Steel", "Concrete Pour", "Rebar Installation", "Formwork",
  "Roofing", "Glazing", "Cladding", "Waterproofing",
  "Electrical Rough-in", "Plumbing Rough-in", "HVAC Installation", "Fire Protection",
  "Interior Finishing", "Painting", "Flooring",
  "Landscaping", "Paving", "Utilities Connection",
  "Inspection", "Testing", "Commissioning"
]

# Construction phases for temporal sequencing within project timeline
phase_order = {
  "Demolition": 1, "Excavation": 1, "Grading": 1,
  "Piling": 2, "Foundation Pour": 2,
  "Structural Steel": 3, "Rebar Installation": 3, "Formwork": 3, "Concrete Pour": 3,
  "Roofing": 4, "Glazing": 4, "Cladding": 4, "Waterproofing": 4,
  "Electrical Rough-in": 5, "Plumbing Rough-in": 5, "HVAC Installation": 5, "Fire Protection": 5,
  "Interior Finishing": 6, "Painting": 6, "Flooring": 6,
  "Landscaping": 7, "Paving": 7, "Utilities Connection": 7,
  "Inspection": 8, "Testing": 8, "Commissioning": 8,
}

# Work order type affinity by project type
project_wo_types = {
  "Bridge":                ["Excavation", "Piling", "Foundation Pour", "Structural Steel", "Concrete Pour",
                            "Rebar Installation", "Formwork", "Waterproofing", "Painting", "Inspection", "Testing"],
  "Stadium":               ["Demolition", "Excavation", "Grading", "Piling", "Foundation Pour", "Structural Steel",
                            "Concrete Pour", "Roofing", "Glazing", "Cladding", "Electrical Rough-in",
                            "Plumbing Rough-in", "HVAC Installation", "Interior Finishing", "Painting",
                            "Flooring", "Landscaping", "Paving", "Inspection", "Commissioning"],
  "Tunnel":                ["Demolition", "Excavation", "Piling", "Concrete Pour", "Rebar Installation",
                            "Formwork", "Waterproofing", "Electrical Rough-in", "Fire Protection",
                            "Utilities Connection", "Inspection", "Testing", "Commissioning"],
  "Hospital":              ["Demolition", "Excavation", "Foundation Pour", "Structural Steel", "Concrete Pour",
                            "Roofing", "Glazing", "Cladding", "Electrical Rough-in", "Plumbing Rough-in",
                            "HVAC Installation", "Fire Protection", "Interior Finishing", "Painting",
                            "Flooring", "Inspection", "Testing", "Commissioning"],
  "Highway":               ["Demolition", "Excavation", "Grading", "Foundation Pour", "Concrete Pour",
                            "Rebar Installation", "Paving", "Utilities Connection", "Landscaping",
                            "Inspection", "Testing"],
  "High-Rise":             ["Demolition", "Excavation", "Piling", "Foundation Pour", "Structural Steel",
                            "Concrete Pour", "Formwork", "Roofing", "Glazing", "Cladding", "Waterproofing",
                            "Electrical Rough-in", "Plumbing Rough-in", "HVAC Installation", "Fire Protection",
                            "Interior Finishing", "Painting", "Flooring", "Inspection", "Commissioning"],
  "Airport Terminal":      ["Demolition", "Excavation", "Grading", "Piling", "Foundation Pour", "Structural Steel",
                            "Concrete Pour", "Roofing", "Glazing", "Cladding", "Electrical Rough-in",
                            "Plumbing Rough-in", "HVAC Installation", "Fire Protection", "Interior Finishing",
                            "Painting", "Flooring", "Paving", "Landscaping", "Inspection", "Commissioning"],
  "Dam":                   ["Excavation", "Grading", "Piling", "Foundation Pour", "Concrete Pour",
                            "Rebar Installation", "Formwork", "Waterproofing", "Utilities Connection",
                            "Inspection", "Testing", "Commissioning"],
  "Rail Station":          ["Demolition", "Excavation", "Foundation Pour", "Structural Steel", "Concrete Pour",
                            "Roofing", "Glazing", "Electrical Rough-in", "Plumbing Rough-in",
                            "Interior Finishing", "Painting", "Flooring", "Paving", "Inspection", "Commissioning"],
  "Convention Center":     ["Demolition", "Excavation", "Foundation Pour", "Structural Steel", "Concrete Pour",
                            "Roofing", "Glazing", "Cladding", "Electrical Rough-in", "Plumbing Rough-in",
                            "HVAC Installation", "Interior Finishing", "Painting", "Flooring", "Landscaping",
                            "Inspection", "Commissioning"],
  "School":                ["Demolition", "Excavation", "Foundation Pour", "Concrete Pour", "Roofing",
                            "Electrical Rough-in", "Plumbing Rough-in", "HVAC Installation",
                            "Interior Finishing", "Painting", "Flooring", "Landscaping", "Inspection"],
  "Water Treatment Plant": ["Excavation", "Grading", "Piling", "Foundation Pour", "Concrete Pour",
                            "Rebar Installation", "Waterproofing", "Electrical Rough-in", "Plumbing Rough-in",
                            "Utilities Connection", "Inspection", "Testing", "Commissioning"],
}

trades = ["General", "Civil", "Structural", "Mechanical", "Electrical", "Plumbing",
          "HVAC", "Fire Protection", "Roofing", "Glazing", "Earthworks", "Finishing"]

wo_type_trade = {
  "Excavation": "Earthworks", "Demolition": "General", "Grading": "Earthworks",
  "Piling": "Structural", "Foundation Pour": "Structural",
  "Structural Steel": "Structural", "Concrete Pour": "Structural",
  "Rebar Installation": "Structural", "Formwork": "Structural",
  "Roofing": "Roofing", "Glazing": "Glazing", "Cladding": "General",
  "Waterproofing": "General",
  "Electrical Rough-in": "Electrical", "Plumbing Rough-in": "Plumbing",
  "HVAC Installation": "HVAC", "Fire Protection": "Fire Protection",
  "Interior Finishing": "Finishing", "Painting": "Finishing", "Flooring": "Finishing",
  "Landscaping": "General", "Paving": "Civil", "Utilities Connection": "Civil",
  "Inspection": "General", "Testing": "General", "Commissioning": "General",
}

# Cost per work order type (mu, sigma for log-normal, lo, hi in thousands)
wo_cost_params = {
  "Excavation":          (3.5, 0.8,  10,  800),
  "Demolition":          (3.2, 0.7,   8,  500),
  "Grading":             (3.0, 0.6,   5,  400),
  "Piling":              (4.0, 0.8,  20, 1200),
  "Foundation Pour":     (4.2, 0.7,  30, 1500),
  "Structural Steel":    (4.5, 0.8,  50, 2000),
  "Concrete Pour":       (3.8, 0.7,  15, 1000),
  "Rebar Installation":  (3.5, 0.7,  10,  800),
  "Formwork":            (3.3, 0.6,   8,  600),
  "Roofing":             (3.5, 0.7,  12,  900),
  "Glazing":             (3.8, 0.8,  15, 1200),
  "Cladding":            (3.5, 0.7,  12,  800),
  "Waterproofing":       (3.0, 0.6,   5,  400),
  "Electrical Rough-in": (3.8, 0.7,  15, 1000),
  "Plumbing Rough-in":   (3.5, 0.7,  10,  800),
  "HVAC Installation":   (4.0, 0.7,  20, 1200),
  "Fire Protection":     (3.3, 0.6,   8,  600),
  "Interior Finishing":  (3.8, 0.7,  15, 1000),
  "Painting":            (2.8, 0.6,   3,  300),
  "Flooring":            (3.2, 0.6,   8,  500),
  "Landscaping":         (2.5, 0.5,   3,  200),
  "Paving":              (3.5, 0.7,  10,  800),
  "Utilities Connection":(3.3, 0.6,   8,  600),
  "Inspection":          (2.0, 0.4,   2,  100),
  "Testing":             (2.5, 0.5,   3,  150),
  "Commissioning":       (3.0, 0.6,   5,  300),
}

# Crew size by work order type (mu, sigma)
crew_params = {
  "Excavation": (15, 5), "Demolition": (20, 7), "Grading": (12, 4),
  "Piling": (25, 8), "Foundation Pour": (30, 10),
  "Structural Steel": (35, 10), "Concrete Pour": (25, 8),
  "Rebar Installation": (20, 6), "Formwork": (18, 5),
  "Roofing": (15, 5), "Glazing": (12, 4), "Cladding": (15, 5),
  "Waterproofing": (10, 3),
  "Electrical Rough-in": (20, 6), "Plumbing Rough-in": (15, 5),
  "HVAC Installation": (18, 5), "Fire Protection": (12, 4),
  "Interior Finishing": (25, 8), "Painting": (10, 3), "Flooring": (12, 4),
  "Landscaping": (8, 3), "Paving": (15, 5), "Utilities Connection": (10, 3),
  "Inspection": (3, 1), "Testing": (5, 2), "Commissioning": (8, 3),
}

# Seasonal weather delay factors by region (Jan-Dec, higher = more delay)
weather_delay_by_region = {
  "Northeast":         [1.8, 1.6, 1.3, 0.8, 0.5, 0.3, 0.3, 0.3, 0.5, 0.8, 1.2, 1.7],
  "Southeast":         [0.6, 0.6, 0.7, 0.8, 1.0, 1.3, 1.4, 1.5, 1.4, 0.8, 0.6, 0.6],
  "Midwest":           [2.0, 1.8, 1.4, 0.9, 0.5, 0.4, 0.3, 0.3, 0.5, 0.8, 1.3, 1.9],
  "Southwest":         [0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 1.5, 1.6, 1.2, 0.5, 0.3, 0.3],
  "West":              [1.0, 1.0, 0.8, 0.5, 0.3, 0.2, 0.1, 0.1, 0.2, 0.4, 0.7, 1.0],
  "Pacific Northwest": [1.5, 1.4, 1.3, 1.1, 0.8, 0.5, 0.3, 0.3, 0.5, 1.0, 1.3, 1.5],
}

# Safety incident base probability by work order type
safety_incident_rates = {
  "Excavation": 0.04, "Demolition": 0.05, "Grading": 0.02,
  "Piling": 0.05, "Foundation Pour": 0.03,
  "Structural Steel": 0.06, "Concrete Pour": 0.04,
  "Rebar Installation": 0.04, "Formwork": 0.03,
  "Roofing": 0.05, "Glazing": 0.03, "Cladding": 0.03,
  "Waterproofing": 0.02,
  "Electrical Rough-in": 0.04, "Plumbing Rough-in": 0.02,
  "HVAC Installation": 0.03, "Fire Protection": 0.02,
  "Interior Finishing": 0.01, "Painting": 0.01, "Flooring": 0.01,
  "Landscaping": 0.01, "Paving": 0.02, "Utilities Connection": 0.02,
  "Inspection": 0.005, "Testing": 0.005, "Commissioning": 0.005,
}

subcontractors_by_trade = {}
for t in trades:
  subcontractors_by_trade[t] = [fake.company() for _ in range(15)]

foremen = [fake.name() for _ in range(200)]

NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
work_orders = []

for i in range(1, NUM_EVENT_RECORDS + 1):
  pid = random.randint(1, NUM_PROJECTS)
  proj = project_lookup[pid]

  available_types = project_wo_types.get(proj.project_type, work_order_types)
  wo_type = random.choice(available_types)

  trade = wo_type_trade.get(wo_type, "General")
  subcontractor = random.choice(subcontractors_by_trade.get(trade, subcontractors_by_trade["General"]))
  foreman_name = random.choice(foremen)

  # Phase-based temporal positioning within project timeline
  phase = phase_order.get(wo_type, 5)
  proj_duration_days = (proj.estimated_completion_date - proj.start_date).days
  if proj_duration_days < 30:
    proj_duration_days = 30

  phase_start_frac = (phase - 1) / 8.0
  phase_end_frac = min(phase / 8.0 + 0.15, 1.0)
  wo_start_offset = int(proj_duration_days * clamp(random.gauss(
    (phase_start_frac + phase_end_frac) / 2, 0.08), phase_start_frac, phase_end_frac))
  sched_start = proj.start_date + timedelta(days=wo_start_offset)

  wo_duration_days = int(clamp(random.lognormvariate(2.5, 0.7), 3, 180))
  sched_end = sched_start + timedelta(days=wo_duration_days)

  # Status depends on dates relative to NOW
  if sched_end < NOW.date() - timedelta(days=14):
    status = random.choices(["Completed", "Delayed", "Cancelled"],
                            weights=[82, 10, 8])[0]
  elif sched_start > NOW.date():
    status = random.choices(["Scheduled", "Cancelled"],
                            weights=[90, 10])[0]
  elif sched_start <= NOW.date() <= sched_end:
    status = random.choices(["In Progress", "Delayed", "Scheduled", "Cancelled"],
                            weights=[60, 20, 12, 8])[0]
  else:
    status = random.choices(["Completed", "In Progress", "Delayed"],
                            weights=[65, 20, 15])[0]

  # Cascade project status to work orders
  if proj.status == "Cancelled":
    status = random.choices(["Cancelled", "Completed"], weights=[80, 20])[0]
  elif proj.status == "On Hold":
    if status not in ["Completed", "Cancelled"]:
      status = random.choices(["Delayed", "Scheduled", "Cancelled"], weights=[50, 30, 20])[0]

  # Actual dates
  if status in ["Completed", "In Progress", "Delayed"]:
    delay_days = int(clamp(random.gauss(2, 5), -5, 30)) if status == "Delayed" \
      else int(clamp(random.gauss(0, 2), -3, 5))
    actual_start = sched_start + timedelta(days=delay_days)
  else:
    actual_start = None

  if status == "Completed":
    overrun = int(clamp(random.gauss(wo_duration_days * 0.1, wo_duration_days * 0.2),
                        -5, wo_duration_days * 0.5))
    actual_end = sched_end + timedelta(days=overrun)
    if actual_end > NOW.date():
      actual_end = NOW.date() - timedelta(days=random.randint(0, 7))
  else:
    actual_end = None

  cp = crew_params.get(wo_type, (15, 5))
  crew_size = int(clamp(random.gauss(cp[0], cp[1]), 2, 80))

  # Estimated cost (log-normal, scaled by project size)
  cost_p = wo_cost_params.get(wo_type, (3.5, 0.7, 10, 800))
  estimated_cost = round(clamp(random.lognormvariate(cost_p[0], cost_p[1]),
                               cost_p[2], cost_p[3]) * 1000, 2)
  size_factor = clamp(proj.contract_value / 500_000_000, 0.5, 3.0)
  estimated_cost = round(estimated_cost * size_factor, 2)

  # Actual cost with overrun tendency (construction is notorious for overruns)
  if status == "Completed":
    overrun_factor = clamp(random.lognormvariate(0.05, 0.15), 0.85, 1.60)
    actual_cost = round(estimated_cost * overrun_factor, 2)
  elif status == "In Progress":
    progress = clamp(random.betavariate(3.0, 2.0), 0.2, 0.9)
    actual_cost = round(estimated_cost * progress * clamp(random.gauss(1.05, 0.1), 0.9, 1.3), 2)
  elif status == "Cancelled":
    actual_cost = round(estimated_cost * clamp(random.betavariate(1.5, 4.0), 0.0, 0.4), 2)
  else:
    actual_cost = None

  # Materials vs labor cost split
  if actual_cost is not None and actual_cost > 0:
    materials_pct = clamp(random.gauss(0.45, 0.10), 0.25, 0.70)
    materials_cost = round(actual_cost * materials_pct, 2)
    labor_cost = round(actual_cost * (1 - materials_pct), 2)
  else:
    materials_cost = None
    labor_cost = None

  # Equipment hours (log-normal, scaled by crew size)
  if status in ["Completed", "In Progress", "Delayed"]:
    equip_hours = round(clamp(random.lognormvariate(3.0, 0.8), 4, 2000) * (crew_size / 15.0), 1)
  else:
    equip_hours = 0.0

  # Safety incidents (rare events, higher for dangerous work types and project types)
  base_rate = safety_incident_rates.get(wo_type, 0.02)
  if proj.project_type in ["Tunnel", "Bridge", "Dam"]:
    base_rate *= 1.5
  if crew_size > 30:
    base_rate *= 1.3
  safety_incidents = 0
  if status in ["Completed", "In Progress", "Delayed"]:
    if random.random() < base_rate:
      safety_incidents = int(clamp(random.expovariate(2.0) + 1, 1, 5))

  # Weather delay hours (seasonal and regional correlation)
  weather_delay = 0.0
  if status in ["Completed", "In Progress", "Delayed"] and actual_start is not None:
    month_idx = actual_start.month - 1
    region_factor = weather_delay_by_region.get(proj.region, [0.5] * 12)[month_idx]
    if random.random() < 0.25 * region_factor:
      weather_delay = round(clamp(random.lognormvariate(2.0, 0.8), 1, 120) * region_factor, 1)

  # Inspection result
  if wo_type in ["Inspection", "Testing", "Commissioning"] and status == "Completed":
    inspection_passed = random.random() > 0.12
  elif status == "Completed":
    inspection_passed = random.random() > 0.08
  else:
    inspection_passed = None

  # Change orders (very common in construction, ~25% of active work orders)
  change_order = 0.0
  if status in ["Completed", "In Progress", "Delayed"]:
    if random.random() < 0.25:
      change_order = round(clamp(random.lognormvariate(2.5, 1.0), 0.5, 500) * 1000 * size_factor, 2)
      if random.random() < 0.1:
        change_order = -change_order

  priority = random.choices(["Low", "Medium", "High", "Critical"],
                            weights=[15, 40, 30, 15])[0]
  if proj.project_type in ["Hospital", "Airport Terminal"]:
    priority = random.choices(["Low", "Medium", "High", "Critical"],
                              weights=[8, 30, 38, 24])[0]

  work_orders.append(Row(
    work_order_id=5000 + i,
    project_id=pid,
    work_order_type=wo_type,
    trade=trade,
    subcontractor_name=subcontractor,
    foreman=foreman_name,
    crew_size=crew_size,
    scheduled_start_date=sched_start,
    scheduled_end_date=sched_end,
    actual_start_date=actual_start,
    actual_end_date=actual_end,
    status=status,
    estimated_cost=estimated_cost,
    actual_cost=actual_cost,
    materials_cost=materials_cost,
    labor_cost=labor_cost,
    equipment_hours=equip_hours,
    safety_incidents=safety_incidents,
    weather_delay_hours=weather_delay,
    inspection_passed=inspection_passed,
    change_order_amount=change_order,
    priority=priority
  ))

wo_df = spark.createDataFrame(work_orders)
wo_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.work_orders")
print(f"\u2714 Created {CATALOG_SCHEMA}.work_orders ({wo_df.count()} rows)")

print("\n--- Projects (sample) ---")
display(projects_df.limit(5))
print("\n--- Work Orders (sample) ---")
display(wo_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
  for col, comment in comments.items():
    spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
  print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.construction.projects", {
  "project_id":                 "Unique identifier for the construction project",
  "project_name":               "Descriptive project name combining city and project descriptor",
  "project_type":               "Type of construction: Bridge, Stadium, Tunnel, Hospital, Highway, High-Rise, Airport Terminal, Dam, Rail Station, Convention Center, School, Water Treatment Plant",
  "client_name":                "Client organization name (generated via Faker)",
  "project_manager":            "Project manager name (generated via Faker)",
  "region":                     "Geographic region: Northeast, Southeast, Midwest, Southwest, West, or Pacific Northwest",
  "city":                       "City where the project is located (31 cities across 6 regions)",
  "status":                     "Project status: Planning, In Progress, Completed, On Hold, or Cancelled",
  "contract_value":             "Total contract value in USD (log-normal distributed, varies by project type from $8M schools to $5B stadiums)",
  "start_date":                 "Project start date (DateType)",
  "estimated_completion_date":  "Estimated completion date based on project-type-specific duration (DateType)",
  "actual_completion_date":     "Actual completion date for completed projects, NULL otherwise. Reflects schedule overruns (DateType)",
  "completion_pct":             "Percentage of project completed (0-100). Beta-distributed for in-progress, correlated with timeline",
  "safety_rating":              "Safety performance rating 0-100 (beta-distributed, higher is better). Reduced for tunnels, dams, and bridges",
  "environmental_impact_score": "Environmental impact score 0-100 (beta-distributed, lower is better). Higher for highways, dams, and tunnels",
  "num_workers_peak":           "Peak workforce count (Gaussian distributed, varies by project type from ~200 for schools to ~3000 for airport terminals)",
  "site_area_sqm":              "Construction site area in square meters (log-normal distributed)",
})

apply_comments(f"{CATALOG}.construction.work_orders", {
  "work_order_id":         "Unique identifier for the work order",
  "project_id":            "Foreign key referencing projects.project_id",
  "work_order_type":       "Type of work: 26 types across 8 construction phases (site prep through commissioning)",
  "trade":                 "Trade discipline: General, Civil, Structural, Mechanical, Electrical, Plumbing, HVAC, Fire Protection, Roofing, Glazing, Earthworks, or Finishing",
  "subcontractor_name":    "Subcontractor company name (generated via Faker, 15 per trade)",
  "foreman":               "Crew foreman name (generated via Faker, 200 foremen pool)",
  "crew_size":             "Number of workers assigned (Gaussian distributed, varies by work order type)",
  "scheduled_start_date":  "Planned start date, positioned within project timeline based on construction phase (DateType)",
  "scheduled_end_date":    "Planned end date (DateType). Duration is log-normal distributed (3-180 days)",
  "actual_start_date":     "Actual start date (DateType). NULL for scheduled/cancelled orders. Includes delay variance",
  "actual_end_date":       "Actual end date (DateType). NULL unless completed. Reflects schedule overruns",
  "status":                "Work order status: Completed, In Progress, Scheduled, Delayed, or Cancelled. Influenced by project status",
  "estimated_cost":        "Estimated cost in USD (log-normal distributed, scaled by project size)",
  "actual_cost":           "Actual cost in USD. Reflects construction cost overrun tendency (85-160% of estimate for completed orders)",
  "materials_cost":        "Materials portion of actual cost (Gaussian split, ~25-70% of actual cost)",
  "labor_cost":            "Labor portion of actual cost (remainder after materials)",
  "equipment_hours":       "Heavy equipment hours used (log-normal distributed, scaled by crew size)",
  "safety_incidents":      "Number of safety incidents (rare; exponential distribution when triggered). Higher rates for structural/demolition work and tunnel/bridge projects",
  "weather_delay_hours":   "Hours lost to weather (log-normal when triggered). Seasonal and regional: worse in winter for Northeast/Midwest, monsoon for Southwest, rain for Pacific Northwest",
  "inspection_passed":     "Whether the work passed inspection (Boolean). ~8-12% failure rate. NULL if not yet inspected",
  "change_order_amount":   "Change order delta in USD. 25% of active work orders have changes (log-normal). 10% are negative (scope reductions)",
  "priority":              "Work order priority: Low, Medium, High, or Critical. Hospital and airport projects skew toward higher priority",
})

print(f"\n\u2705 All column comments applied for construction schema")

spark.sql(f"ALTER TABLE {CATALOG}.construction.projects ALTER COLUMN project_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.construction.projects ADD CONSTRAINT pk_projects PRIMARY KEY (project_id)")
spark.sql(f"ALTER TABLE {CATALOG}.construction.work_orders ALTER COLUMN work_order_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.construction.work_orders ADD CONSTRAINT pk_work_orders PRIMARY KEY (work_order_id)")
spark.sql(f"ALTER TABLE {CATALOG}.construction.work_orders ADD CONSTRAINT fk_work_orders_project_id FOREIGN KEY (project_id) REFERENCES {CATALOG}.construction.projects(project_id)")
print(f"\u2714 PK/FK constraints applied for construction schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.construction') IS
'Construction sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `projects` (~2K rows, PK: project_id). Event table: `work_orders` (100K-500K rows, PK: work_order_id, FK: project_id → projects). Key features: 12 project types (bridges, stadiums, tunnels, hospitals, highways, high-rises, airports, dams, rail stations, convention centers, schools, water treatment plants), phase-sequenced work orders, cost overrun modeling, seasonal/regional weather delays, safety incident tracking, change order patterns, materials/labor cost splits.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`construction` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"\u2714 RemoveAfter tag applied to construction schema ({remove_after_value})")